# 01. 사출 공정·시간 EDA

S14 설비의 CN7·RG3 원단위 데이터를 시간·좌우·불량 사유와 연결합니다. **모델 학습 노트북이 아닙니다.** [설명 보고서](../artifacts/process_eda_v1/report.md)를 함께 읽어 주세요.

커널: **Python (kamp-base)**. 위에서 아래로 실행합니다. 기존 baseline은 `../archive/2026-09-29_tabular_v1/`에 있습니다.

In [ ]:
from pathlib import Path
from datetime import datetime
import sys, json, subprocess, zipfile
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

PROJECT = None
for base in [Path.cwd(), *Path.cwd().parents]:
    for candidate in (base, base / "1.injection-molding"):
        if (candidate / "scripts/process_eda.py").exists():
            PROJECT = candidate
            break
    if PROJECT is not None:
        break
if PROJECT is None:
    raise FileNotFoundError("현재 사출 프로젝트에서 노트북을 열어 주세요.")
RESULTS = PROJECT / "artifacts/process_eda_v1"
RUN_EDA = False  # True: 원본 ZIP에서 새 폴더로 재생성. 모델 학습은 하지 않습니다.
if RUN_EDA:
    RESULTS = PROJECT / "artifacts/process_eda_runs" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    subprocess.run([sys.executable, str(PROJECT / "scripts/process_eda.py"), "--output", str(RESULTS)], check=True)
protocol = json.loads((RESULTS / "protocol.json").read_text())
print("결과:", RESULTS)
print("모델 학습 여부:", protocol["models_trained"])

## 1. 실제 기록과 불량은 언제 있었나

중복 ID를 제외한 관측 건수입니다. 공장 전체 생산량이나 불량률을 의미하지 않습니다. 빈 날짜는 관측이 없는 것이며 생산 중단으로 확정할 수 없습니다.

In [ ]:
daily = pd.read_csv(RESULTS / "daily_counts.csv")
display(daily.groupby("product")[["records", "defects"]].sum())
reasons = pd.read_csv(RESULTS / "daily_reasons.csv")
display(reasons[reasons.Reason.notna()])
for name in ["cn7_defect_days.png", "rg3_defect_days.png"]:
    display(Image(filename=str(RESULTS / "figures" / name), width=1000))

## 2. 전체 기간의 원단위 공정값

파랑·주황 점은 좌우, 빨간 표시는 불량입니다. `Gas`=가스, `Incomplete fill`=미성형, `Startup allowance`=초기허용불량입니다. 관측 공백을 선으로 연결하지 않았습니다. 점선은 뒤에서 검토할 **탐색용** 시간 경계이며 최종 평가 분할은 아닙니다.

CN7은 초기허용불량의 충전 시간·압력 변화가 눈에 띕니다. 반면 같은 종류의 미성형도 날짜에 따라 편차 방향이 달라질 수 있습니다.

In [ ]:
for name in ["cn7_timeline.png", "rg3_timeline.png"]:
    display(Image(filename=str(RESULTS / "figures" / name), width=1100))

## 3. 원하는 날짜·좌우로 확대

다음 셀의 `PRODUCT`, `SIDE`, `DAY`를 바꾸고 실행하세요. 기본은 CN7 RH의 초기허용불량이 있던 10/27입니다. RG3 LH의 11/04도 확인해 보세요.

이 확대는 사후 탐색이며 해당 날짜를 좋은 평가 사례로 골랐다는 뜻이 아닙니다. 원본 ZIP이 없으면 이 셀만 건너뜁니다.

In [ ]:
PRODUCT = "CN7"
SIDE = "RH"
DAY = "2020-10-27"
source = PROJECT / "sources/04. Dataset_Molding.zip"
if not source.exists():
    display(Markdown("원본 ZIP이 없어 날짜 확대를 생략합니다. 저장된 요약과 그림은 확인할 수 있습니다."))
else:
    with zipfile.ZipFile(source) as z:
        with z.open("dataset/labeled_data.csv") as f:
            raw = pd.read_csv(f).drop_duplicates("_id")
    raw["timestamp"] = pd.to_datetime(raw["TimeStamp"])
    focus = raw[(raw.EQUIP_CD == "S14") & raw.PART_NAME.str.startswith(PRODUCT)
                & raw.PART_NAME.str.strip().str.endswith(SIDE)
                & raw.timestamp.dt.strftime("%Y-%m-%d").eq(DAY)].sort_values("timestamp")
    if focus.empty:
        print("선택한 조건의 관측이 없습니다.")
    else:
        fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
        styles = {"가스": ("x", "Gas"), "미성형": ("^", "Incomplete fill"), "초기허용불량": ("s", "Startup allowance")}
        for ax, feature in zip(axes, ["Filling_Time", "Max_Injection_Pressure", "Mold_Temperature_3"]):
            normal = focus[focus.PassOrFail.eq("Y")]
            ax.scatter(normal.timestamp, normal[feature], s=12, alpha=.5, label="Normal")
            for reason, (marker, label) in styles.items():
                defects = focus[focus.Reason.eq(reason)]
                if len(defects):
                    ax.scatter(defects.timestamp, defects[feature], marker=marker, s=55, color="crimson", label=label)
            ax.set_ylabel(feature)
            ax.grid(alpha=.2)
            ax.legend(fontsize=8)
        axes[0].set_title(f"{PRODUCT} {SIDE} / {DAY} (descriptive)")
        fig.autofmt_xdate();plt.tight_layout();plt.show()
        print("관측 기록:", len(focus), " / defects:", int(focus.PassOrFail.eq("N").sum()))

## 4. 같은 날·같은 쪽 양품과 비교

전체 양품과 비교하면 날짜·좌우의 영향이 섞일 수 있어 범위를 좁혔습니다. 그래도 인과관계는 아닙니다. 사유별 표본 수(`defect_n`)를 꼭 함께 보세요. IQR은 가운데 50% 범위이며, IQR 대비 차이가 크다고 불량 확률이 높다고 단정하지 않습니다.

In [ ]:
comparison = pd.read_csv(RESULTS / "same_day_side_comparison.csv")
SELECT_PRODUCT = "CN7"
SELECT_FEATURES = ["Filling_Time", "Max_Injection_Pressure", "Mold_Temperature_3"]
view = comparison[comparison["product"].eq(SELECT_PRODUCT) & comparison.feature.isin(SELECT_FEATURES)]
display(view[["product", "side", "date", "reason", "feature", "normal_n", "defect_n", "normal_median", "defect_median", "raw_median_difference"]].round(4))

## 5. 공백과 생산 재시작을 혼동하지 않기

10분 넘게 관측이 비면 새 구간으로 나눴습니다. 이 경계는 실제 재시작 로그가 아닙니다. CN7 초기허용불량은 구간 시작 후 약 28~37분에 관측되어, 이름만 보고 '첫 10분 불량'으로 라벨링할 수 없습니다.

In [ ]:
display(pd.read_csv(RESULTS / "gap_sensitivity.csv"))
display(pd.read_csv(RESULTS / "segment_start_counts.csv"))

## 6. 미라벨은 정상 데이터와 같은 분포인가

아래 후보 풀은 학습 종료 시각 이전이고, 라벨 자료와 겹치는 설비·제품·시각 후보를 좌우 모두 제외했습니다. 정상이라고 확인된 데이터는 아닙니다.

CN7 LH는 미라벨 후보에서 사이클 중앙값 36.60초, 라벨 학습 구간에서 59.52초로 다릅니다. 같은 제품·설비라도 운전 조건과 센서 사용이 달라질 수 있으므로 전부 섞어 정상 기준을 만들지 않습니다.

In [ ]:
display(pd.read_csv(RESULTS / "unlabeled_eligibility.csv"))
context = pd.read_csv(RESULTS / "distribution_context.csv")
display(context[context.source.ne("labeled_all_descriptive") & context.feature.isin(["Cycle_Time", "Max_Injection_Pressure", "Mold_Temperature_3"])].round(3))

## 7. 시간으로 학습·검증·평가를 나누면

관측 날짜의 앞 60%·중간 20%·뒤 20%이며 **행 개수 비율은 아닙니다.** 라벨을 기준으로 경계를 선택하지 않았지만, 이미 전체를 살펴본 EDA이므로 독립 최종 테스트라고 주장할 수 없습니다.

- CN7 검증 구간의 불량이 0개: 불량 Recall/F1 기준 임계값 선택 불가.
- CN7 평가 구간의 불량이 1개: Recall이 0% 또는 100%로 크게 흔들림.
- RG3 평가 구간의 전체 관측이 3개이고 모두 양품: 불량 탐지율 평가 불가.

따라서 이 3분할을 최종 분할로 채택하지 않습니다. 과거→미래의 날짜별 개발 백테스트와 추가 독립 평가 자료가 필요합니다.

In [ ]:
split = pd.read_csv(RESULTS / "chronological_feasibility.csv")
split["period"] = pd.Categorical(split["period"], ["train", "validation", "test"], ordered=True)
display(split.sort_values(["product", "period"]))
display(pd.read_csv(RESULTS / "period_reasons.csv"))
print(json.dumps(json.loads((RESULTS / "verification.json").read_text()), ensure_ascii=False, indent=2))

## 다음 작업

**미라벨의 운전 조건·센서 사용 구간을 먼저 조사하고, 개발용 시간 백테스트를 정한 뒤 단순 이상 탐지를 시작**합니다. AE와 기존 분류기는 같은 시간 분할에서 비교합니다.

이번 결과에서 '이 열이 원인이다', '이 값을 넘으면 특정 불량이다', '이상 탐지가 분류기보다 좋다'는 결론은 내리지 않았습니다. [보고서](../artifacts/process_eda_v1/report.md)에 근거와 한계를 정리했습니다.